# 00 — Preprocessing BraTS 2020

Part A computes the tumor volume and localisation tables from the segmentations (about 5 minutes).
Part B (optional) rebuilds the whole preprocessed dataset from the raw BraTS archive (about 1-2 hours on Colab).
The preprocessed dataset `BraTS_2020_Data_preprocessing_nii_FIXED.zip` is used by every other notebook and by the app.

In [ ]:
# --- Setup: mount Drive + install the BrainTumorLab code (run this cell first) ---
from google.colab import drive
drive.mount("/content/drive")

import os, sys
REPO_URL = ""   # e.g. "https://github.com/<your-account>/BrainTumorLab.git" ; empty = use the zip on Drive
ZIP_ON_DRIVE = "/content/drive/MyDrive/miniProjetAI/BrainTumorLab.zip"
if not os.path.isfile("/content/BrainTumorLab/pyproject.toml"):
    import shutil
    shutil.rmtree("/content/BrainTumorLab", ignore_errors=True)
    if REPO_URL:
        !git clone -q {REPO_URL} /content/BrainTumorLab
    else:
        !rm -rf /content/btl_tmp && unzip -q -o "{ZIP_ON_DRIVE}" -d /content/btl_tmp
        _src = "/content/btl_tmp/BrainTumorLab" if os.path.isfile("/content/btl_tmp/BrainTumorLab/pyproject.toml") else "/content/btl_tmp"
        shutil.move(_src, "/content/BrainTumorLab")
!pip install -q -e /content/BrainTumorLab
sys.path.insert(0, "/content/BrainTumorLab/src")

import torch
from braintumorlab.config import Paths, seed_everything
seed_everything(0)
paths = Paths("/content/drive/MyDrive/miniProjetAI")
DATA_ROOT = paths.ensure_data_extracted()     # unzips BraTS_2020_Data_preprocessing_nii_FIXED.zip once
print("data:", DATA_ROOT, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")

## A. Tumor volume and localisation tables

Volumes are computed from the label map (1 = NCR, 2 = ED, 4 = ET; 1 voxel = 1 mm³). Localisation uses the
NIfTI orientation (affine) and the distance to the brain surface. Output: `recap_stats_volumes_2020_v2.csv`
and `tumor_localization_2020_v2.csv`.

In [ ]:
from braintumorlab.analysis import volumes_table, localization_table
vol = volumes_table(DATA_ROOT)
vol.to_csv(paths.base / "recap_stats_volumes_2020_v2.csv", index=False)
vol[["vol_WT_ml", "vol_TC_ml", "vol_ET_ml", "vol_ED_ml", "vol_NCR_ml"]].describe().round(1)

In [ ]:
loc = localization_table(DATA_ROOT)
loc.to_csv(paths.base / "tumor_localization_2020_v2.csv", index=False)
loc[["hemisphere", "anterior_posterior", "depth_category"]].value_counts().head(10)

## B. (Optional) rebuild the preprocessed dataset

The z-score is computed **inside the brain** (background = 0), and a `brain_mask.nii.gz`
is saved for each patient. Needs `archive.zip` (Kaggle BraTS 2020) in `MyDrive/miniProjetAI/`.

In [ ]:
from braintumorlab.preprocessing import ensure_brats2020_extracted, run_pipeline
RAW = ensure_brats2020_extracted(str(paths.base / "archive.zip"), "/content")
OUT = "/content/BraTS_2020_Data_preprocessing_nii_V5"
df_meta = run_pipeline(RAW, OUT, max_patients=None, overwrite=True)   # max_patients=5 for a quick test
df_meta.head()

In [ ]:
import shutil
shutil.make_archive("/content/BraTS_2020_Data_preprocessing_nii_V5", "zip", OUT)
shutil.copy("/content/BraTS_2020_Data_preprocessing_nii_V5.zip", paths.base)
shutil.copy(f"{OUT}/patients_metadata_2020.csv", paths.base / "patients_metadata_2020_v5.csv")
print("To use V5 everywhere: set BTL_DATA_ZIP / BTL_DATA_ROOT, or rename the zip.")